# Notebook 7 — Full Motor Imagery Classification System

---

## Section 0 — What we're building on

### Prior concepts this notebook depends on

You have built three complete classifiers for EEG motor imagery:

- **CSP + LDA** (Notebook 4) — closed-form spatial filtering + linear discriminant. Your baseline: **κ = 0.411 ± 0.192** across 9 subjects.
- **EEGNet** (Notebook 5) — compact CNN with depthwise/separable convolutions. Result: **κ = 0.419 ± 0.179**.
- **EEG Conformer** (Notebook 6) — CNN-Transformer hybrid with patch tokenization and self-attention. Result: **κ = 0.406 ± 0.153**.

All three produced nearly identical performance. You understand why: with only ~690 training windows per subject, the deep models can't reliably outperform CSP's strong inductive bias.

### What this notebook adds

**Subject-independent evaluation.** Every result so far was *subject-dependent* — train on subject A01, test on subject A01. This is the calibration paradigm: each user sits through a training session before the BCI works. The capstone introduces *subject-independent* (leave-one-subject-out) evaluation: train on 8 subjects, test on the held-out 1. This is the paradigm that matters for consumer BCI products — no calibration, plug-and-play.

**Model explainability via saliency maps.** You'll compute input gradients (backprop the class prediction to the raw EEG input) to visualize *where* the models look. If EEGNet learned something neurophysiologically meaningful, its saliency should concentrate over sensorimotor channels (C3/C4) during the active imagery period — the same pattern you saw in your ERD maps.

**Systematic comparison.** This notebook produces the final results table: 3 models × 2 paradigms × 9 subjects, with accuracy, kappa, and confusion matrices. This is the deliverable for your project report.

### Why this matters for deployed BCIs

The gap between subject-dependent and subject-independent performance is the single most important number in BCI product development. A system that achieves κ = 0.5 with calibration but κ = 0.1 without it is unusable as a consumer product. The goal is to minimize this gap — and this is where deep learning models, trained on pooled multi-subject data, should finally justify their complexity over CSP.

## Imports and setup

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from scipy.linalg import eigh
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import sys

# Import preprocessing pipeline
sys.path.insert(0, '../4.data preprocessing')
from run_pipeline import run_pipeline

DATA_DIR = '../mne_data/bci_iv_2a'

SUBJECTS = [f'A0{i}' for i in range(1, 10)]
CLASS_NAMES = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']
N_CLASSES = 4
N_CHANNELS = 22
N_TIMEPOINTS = 250
FS = 250  # sampling rate in Hz

# Channel names for Dataset 2a (10-20 system, 22 EEG channels)
CH_NAMES = ['Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4', 'C5', 'C3', 'C1', 'Cz',
            'C2', 'C4', 'C6', 'CP3', 'CP1', 'CPz', 'CP2', 'CP4', 'P1', 'Pz',
            'P2', 'POz']

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (12, 5)

In [2]:
# Device configuration
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f"Using device: {DEVICE}")

Using device: mps


In [ ]:
# ⚠️  COMPUTE NOTE
# This capstone trains EEGNet and EEG Conformer under two paradigms:
#   - Subject-dependent: 9 models × ~300 epochs each (same as before)
#   - Subject-independent (LOSO): 9 folds × ~300 epochs, with ~8× more training data per fold
#
# Subject-independent training is substantially slower because each fold
# trains on ~5,500 windows instead of ~690. Total compute is roughly 10×
# the single-paradigm run.
#
# Recommended: Google Colab T4 or better.
# Set Runtime → Change runtime type → GPU in Colab.
#
# All results are cached to disk — once computed, re-running is instant.

## Part 1 — Load all subjects

We'll load data for all 9 subjects upfront. Every subject's data lives in its own dict from `run_pipeline()`. We store them in a master dict keyed by subject ID.

In [3]:
# Load all 9 subjects
all_data = {}
for subj in SUBJECTS:
    data = run_pipeline(subj, DATA_DIR)
    all_data[subj] = data
    print(f"{subj}: train={data['X_train'].shape[0]} windows, test={data['X_test'].shape[0]} windows")

total_train = sum(all_data[s]['X_train'].shape[0] for s in SUBJECTS)
total_test  = sum(all_data[s]['X_test'].shape[0]  for s in SUBJECTS)
print(f"\nTotal: {total_train} train + {total_test} test = {total_train + total_test} windows")

A01: train=690 windows, test=174 windows
A02: train=690 windows, test=174 windows
A03: train=690 windows, test=174 windows
A04: train=690 windows, test=174 windows
A05: train=690 windows, test=174 windows
A06: train=687 windows, test=174 windows
A07: train=690 windows, test=174 windows
A08: train=687 windows, test=174 windows
A09: train=684 windows, test=174 windows

Total: 6198 train + 1566 test = 7764 windows


## Part 2 — Re-implement all three models

You're re-implementing each model from scratch, just as you did in the individual notebooks. The architecture and hyperparameters are identical — we're not changing the models, only the evaluation paradigm.

### 2.1 CSP + LDA pipeline

The CSP functions you need:
- `trial_covariances(X)` — normalized spatial covariances, shape `(N, C, C)`
- `fit_binary_csp(X0, X1, n_components)` — generalized eigenvalue decomposition
- `fit_csp_ovr(X, y, n_components, n_classes)` — one-vs-rest wrapper
- `csp_features(X, W)` — log-variance feature extraction

These are the same functions from Notebook 4. Re-implement them here.

> **Why re-implement instead of importing?** Each notebook is self-contained. In a project report or portfolio, someone reading this notebook should see every component. Also, this is your last chance to write these from memory — if you can do it without looking back, you've truly internalized CSP.

In [4]:
# ============================================================
# CSP utility functions — re-implement from Notebook 4
# ============================================================

def trial_covariances(X):
    """
    Compute normalized spatial covariance for each trial.

    Args:
        X: np.ndarray, shape (n_trials, C, T)

    Returns:
        covs: np.ndarray, shape (n_trials, C, C)
              Each matrix is (X_i @ X_i.T) / trace(X_i @ X_i.T).
    """
    # YOUR CODE HERE
    covs = X @ X.transpose(0,2,1)
    traces = np.trace(covs,axis1=1,axis2=2)
    return covs / traces[:,None,None]


def fit_binary_csp(X0, X1, n_components=4):
    """
    Fit binary CSP via generalized eigenvalue decomposition.

    Args:
        X0: np.ndarray, shape (n_trials_0, C, T). Trials of class 0.
        X1: np.ndarray, shape (n_trials_1, C, T). Trials of class 1.
        n_components: int (even). Spatial filters to keep.

    Returns:
        W: np.ndarray, shape (n_components, C)
    """
    # YOUR CODE HERE
    assert n_components % 2 == 0, "n_components must be even"
    cov_0 = trial_covariances(X0).mean(axis=0)
    cov_1 = trial_covariances(X1).mean(axis=0)
    composite = cov_0+cov_1
    composite += np.eye(composite.shape[0]) * 1e-6
    eigenvalues,eigenvectors = eigh(cov_0,composite)
    eigenvalues  = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]
    k = n_components // 2
    selected = np.concatenate([eigenvectors[:, :k], eigenvectors[:, -k:]], axis=1)  # (C, n_components)
    W = selected.T
    return W


def fit_csp_ovr(X, y, n_components=4, n_classes=4):
    """
    Multi-class CSP using one-vs-rest.

    Args:
        X: np.ndarray, shape (n_trials, C, T)
        y: np.ndarray, shape (n_trials,)
        n_components: int, filters per binary problem.
        n_classes: int.

    Returns:
        W: np.ndarray, shape (n_classes * n_components, C)
    """
    # YOUR CODE HERE
    filters = []
    
    for c in range(n_classes):
        X_c    = X[y == c]           # trials for class c
        X_rest = X[y != c]           # all other trials
        W_c    = fit_binary_csp(X_c, X_rest, n_components)  # (n_components, C)
        filters.append(W_c)
    
    W = np.concatenate(filters, axis=0)  # (n_classes * n_components, C)
    return W


def csp_features(X, W):
    """
    Apply CSP filters and extract log-variance features.

    Args:
        X: np.ndarray, shape (n_trials, C, T)
        W: np.ndarray, shape (n_filters, C)

    Returns:
        features: np.ndarray, shape (n_trials, n_filters)
    """
    # YOUR CODE HERE
    Y = X.transpose(0, 2, 1) @ W.T 
    Y = Y.transpose(0, 2, 1)
    var = np.var(Y,axis = 2)
    features = np.log(var)
    return features

In [5]:
# Sanity check — CSP on A01
_data = all_data['A01']
_W = fit_csp_ovr(_data['X_train'], _data['y_train'], n_components=4, n_classes=4)
assert _W.shape == (16, 22), f"Expected (16, 22), got {_W.shape}"

_feat = csp_features(_data['X_train'], _W)
assert _feat.shape == (_data['X_train'].shape[0], 16), f"Expected ({_data['X_train'].shape[0]}, 16), got {_feat.shape}"

# Verify features discriminate: class means should differ
_class_means = np.array([_feat[_data['y_train'] == c].mean(axis=0) for c in range(4)])
_max_dist = np.max(np.ptp(_class_means, axis=0))
assert _max_dist > 0.5, f"CSP features don't discriminate: max class-mean distance = {_max_dist:.3f}"
print(f"✓ CSP pipeline works. Feature shape: {_feat.shape}, max class-mean distance: {_max_dist:.3f}")

✓ CSP pipeline works. Feature shape: (690, 16), max class-mean distance: 1.718


### 2.2 EEGNet

Re-implement the same EEGNet architecture from Notebook 5. Same hyperparameters:
- `F1=8, D=2, F2=16, kern_len=125, sep_kern=16, pool1=4, pool2=8, dropout=0.5`

Also re-implement the `EEGDataset`, `train_one_epoch`, and `evaluate` functions.

In [6]:
class EEGDataset(Dataset):
    """
    Wraps EEG data for PyTorch DataLoader.

    Args:
        X: np.ndarray, shape (N, C, T)
        y: np.ndarray, shape (N,)

    __getitem__ returns:
        x: FloatTensor, shape (1, C, T) — with singleton kernel dim
        label: LongTensor, scalar
    """
    def __init__(self, X, y):
        # YOUR CODE HERE
        self.X = torch.tensor(X,dtype=torch.float32)
        self.y = torch.tensor(y,dtype=torch.long)
    
    def __len__(self):
        # YOUR CODE HERE
        return len(self.X)
    
    def __getitem__(self, idx):
        # YOUR CODE HERE
        return self.X[idx].unsqueeze(0),self.y[idx]

In [7]:
class EEGNet(nn.Module):
    """
    EEGNet: A compact convolutional neural network for EEG-based BCIs.

    Architecture:
        Block 1: Temporal Conv → BN → Depthwise Conv → BN → ELU → AvgPool → Dropout
        Block 2: Separable Conv (Depthwise + Pointwise) → BN → ELU → AvgPool → Dropout
        Classifier: Flatten → Linear

    Args:
        n_channels: int (22)
        n_timepoints: int (250)
        n_classes: int (4)
        F1: int, temporal filters (8)
        D: int, depth multiplier (2)
        F2: int, pointwise filters (default F1*D=16)
        kern_len: int, temporal kernel (125)
        sep_kern: int, separable conv kernel (16)
        pool1: int, first pooling (4)
        pool2: int, second pooling (8)
        dropout: float (0.5)

    forward(x):
        x: (B, 1, C, T)
        returns: (B, n_classes) — logits
    """
    def __init__(self, n_channels=22, n_timepoints=250, n_classes=4,
                 F1=8, D=2, F2=None, kern_len=125, sep_kern=16,
                 pool1=4, pool2=8, dropout=0.5):
        super().__init__()
        if F2 is None:
            F2 = F1 * D

        # ============================================================
        # Block 1 — Temporal + Spatial filtering
        # YOUR CODE HERE
        # ============================================================
        self.F1 = F1
        self.D = D
        self.F2 = F2
        self.block1 = nn.Sequential(
            nn.Conv2d(1,F1,kernel_size=(1,kern_len), padding=(0, kern_len//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1,F1*D,(n_channels,1),groups = F1,bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1,pool1)),
            nn.Dropout2d(dropout)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1*D,F1*D,kernel_size=(1,sep_kern),groups=F1*D, padding=(0,sep_kern//2), bias=False),
            nn.Conv2d(F1*D,F2,(1,1),bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1,pool2)),
            nn.Dropout2d(dropout)
        )
        _flat = F2 * (n_timepoints // (pool1 * pool2))

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(_flat,n_classes)
        )

    def forward(self, x):
        # YOUR CODE HERE
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)

In [8]:
# Sanity check — EEGNet
_model = EEGNet().to(DEVICE)
with torch.no_grad():
    _x = torch.randn(4, 1, 22, 250).to(DEVICE)
    _out = _model(_x)
assert _out.shape == (4, 4), f"Expected (4,4), got {_out.shape}"

_loss = F.cross_entropy(_out, torch.zeros(4, dtype=torch.long).to(DEVICE))
assert abs(_loss.item() - 1.386) < 0.3, f"Loss at init should be ≈log(4), got {_loss.item():.3f}"
print(f"✓ EEGNet: output {_out.shape}, init loss {_loss.item():.3f} (expect ≈1.386)")

_n_params = sum(p.numel() for p in _model.parameters())
print(f"  Parameters: {_n_params:,}")
del _model, _x, _out

✓ EEGNet: output torch.Size([4, 4]), init loss 1.525 (expect ≈1.386)
  Parameters: 2,396


### 2.3 EEG Conformer

Re-implement the Conformer from Notebook 6. Same architecture:
- PatchEmbedding: `F1=40, D=1, kern_len=25, pool_size=75, pool_stride=15`
- TransformerBlock: `d_model=40, n_heads=10, ff_ratio=3`
- `n_layers=6, dropout=0.5`

In [9]:
class PatchEmbedding(nn.Module):
    """
    CNN front-end: raw EEG (B, 1, C, T) → patch tokens (B, T', d_model).

    Args:
        n_channels: int (22)
        F1: int, temporal filters (40)
        D: int, depth multiplier (1)
        kern_len: int, temporal kernel (25)
        pool_size: int, pooling window (75)
        pool_stride: int, pooling stride (15)
        dropout: float (0.5)

    forward(x):
        x: (B, 1, C, T)
        returns: (B, T', d_model) where d_model = F1*D
    """
    def __init__(self, n_channels=22, F1=40, D=1, kern_len=25,
                 pool_size=75, pool_stride=15, dropout=0.5):
        super().__init__()
        self.d_model = F1 * D
        # YOUR CODE HERE
        self.cnn = nn.Sequential(
            nn.Conv2d(1,F1,(1,kern_len),padding = (0,kern_len//2),bias=False),
            nn.BatchNorm2d(F1),
            nn.ELU(),
            nn.Conv2d(F1,F1*D,(n_channels,1),groups=F1,bias=False),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.Dropout(dropout),
        )
        self.pool = nn.AvgPool1d(pool_size,stride=pool_stride)

        self.pos_embed = nn.Parameter(torch.zeros(1, 64, self.d_model))

    def forward(self, x):
        # YOUR CODE HERE
        x = self.cnn(x)
        x = x.squeeze(2)
        x = self.pool(x) 
        x = x.transpose(1,2)
        seq_len = x.size(1)
        return x + self.pos_embed[:, :seq_len, :]


class TransformerBlock(nn.Module):
    """
    Pre-LN Transformer encoder layer.

    Args:
        d_model: int (40)
        n_heads: int (10)
        ff_ratio: int (3)
        dropout: float (0.5)

    forward(x):
        x: (B, T', d_model)
        returns: (output, attn_weights)
    """
    def __init__(self, d_model=40, n_heads=10, ff_ratio=3, dropout=0.5):
        super().__init__()
        # YOUR CODE HERE
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model,n_heads,dropout,batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model,d_model*ff_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model*ff_ratio,d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # YOUR CODE HERE
        residual = x
        x = self.ln1(x)
        x,attn_weights = self.attn(x,x,x)
        x = self.dropout(x)
        x = x+ residual
        residual = x
        x = self.ln2(x)
        x = self.ffn(x)
        x += residual
        return x, attn_weights


class EEGConformer(nn.Module):
    """
    EEG Conformer: CNN front-end + Transformer encoder + classifier.

    Args:
        n_channels: int (22)
        n_timepoints: int (250)
        n_classes: int (4)
        F1: int (40), D: int (1), kern_len: int (25)
        pool_size: int (75), pool_stride: int (15)
        n_heads: int (10), n_layers: int (6), ff_ratio: int (3)
        dropout: float (0.5)

    forward(x):
        x: (B, 1, C, T)
        returns: logits (B, n_classes)
    """
    def __init__(self, n_channels=22, n_timepoints=250, n_classes=4,
                 F1=40, D=1, kern_len=25,
                 pool_size=75, pool_stride=15,
                 n_heads=10, n_layers=6, ff_ratio=3, dropout=0.5):
        super().__init__()
        # YOUR CODE HERE
        d_model = F1 * D
        self.d_model = d_model
        self.n_layers = n_layers
        
        # Compute sequence length after pooling
        self.seq_len = (n_timepoints - pool_size) // pool_stride + 1
        self.patch_embed = PatchEmbedding(n_channels,F1,D,kern_len,pool_size,pool_stride,dropout)
        self.pos_embed = nn.Parameter(torch.zeros((1,self.seq_len,d_model)))
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(d_model,n_heads,ff_ratio,dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model,n_classes)

    def forward(self, x):
        # YOUR CODE HERE
        tokens = self.patch_embed(x)
        tokens = tokens + self.pos_embed
        tokens = self.pos_drop(tokens)
        for block in self.blocks:
            tokens,_ = block(tokens)
        tokens = self.norm(tokens)
        pooled =tokens.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

In [10]:
# Sanity check — EEG Conformer
_model = EEGConformer().to(DEVICE)
with torch.no_grad():
    _x = torch.randn(4, 1, 22, 250).to(DEVICE)
    _out = _model(_x)
assert _out.shape == (4, 4), f"Expected (4,4), got {_out.shape}"

_loss = F.cross_entropy(_out, torch.zeros(4, dtype=torch.long).to(DEVICE))
assert abs(_loss.item() - 1.386) < 0.5, f"Loss at init should be ≈log(4), got {_loss.item():.3f}"
print(f"✓ Conformer: output {_out.shape}, init loss {_loss.item():.3f}")

_n_params = sum(p.numel() for p in _model.parameters())
print(f"  Parameters: {_n_params:,}")
del _model, _x, _out

✓ Conformer: output torch.Size([4, 4]), init loss 1.394
  Parameters: 104,204


### 2.4 Training and evaluation utilities

Re-implement `train_one_epoch` and `evaluate` — same as Notebooks 5/6.

In [11]:
def train_one_epoch(model, loader, optimizer, device):
    """
    Train for one epoch.

    Args:
        model: nn.Module
        loader: DataLoader
        optimizer: torch.optim.Optimizer
        device: torch.device

    Returns:
        avg_loss: float
    """
    # YOUR CODE HERE
    model.train()
    total_loss = 0
    for X,y in loader:
        X,y = X.to(device),y.to(device)
        logits = model(X)
        optimizer.zero_grad()
        loss = F.cross_entropy(logits,y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(loader)


def evaluate_model(model, loader, device):
    """
    Evaluate the model.

    Args:
        model: nn.Module
        loader: DataLoader
        device: torch.device

    Returns:
        accuracy: float
        kappa: float
        all_preds: np.ndarray
        all_labels: np.ndarray
    """
    # YOUR CODE HERE
    model.eval()
    all_preds = []
    all_labels =[]
    with torch.no_grad():
        for X,y in loader:
            X,y = X.to(device),y.to(device)
            logits = model(X)
            preds = torch.argmax(logits,dim= 1)
            all_preds.append(preds)
            all_labels.append(y)
    all_preds  = torch.cat(all_preds).cpu().numpy()
    all_labels = torch.cat(all_labels).cpu().numpy()
    correct = (all_preds == all_labels).sum().item()
    total = len(all_preds)
    accuracy = correct/total
    kappa = cohen_kappa_score(all_labels,all_preds)
    return accuracy,kappa,all_preds,all_labels


def evaluate_csp(y_true, y_pred):
    """
    Compute accuracy and kappa for CSP (no model, just arrays).

    Args:
        y_true, y_pred: np.ndarray, shape (N,)

    Returns:
        dict with 'accuracy' and 'kappa'
    """
    # YOUR CODE HERE
    acc = np.sum(y_true==y_pred)/len(y_pred)
    kappa = cohen_kappa_score(y_true,y_pred,labels=None,weights=None,sample_weight=None)
    return {'accuracy':acc,'kappa':kappa}

---

## Part 3 — Subject-dependent evaluation (baseline replication)

Before introducing the new paradigm, replicate your previous results to confirm everything works. Train each model on each subject's training data, test on that subject's test data.

This also establishes the caching structure — each model × subject result is saved, so you never re-train.

### 3.1 CSP + LDA — subject-dependent

In [12]:
import pickle

CSP_SD_PATH = 'metrics/capstone_csp_lda_subject_dependent.pkl'

if os.path.exists(CSP_SD_PATH):
    with open(CSP_SD_PATH, 'rb') as f:
        csp_sd_results = pickle.load(f)
    print("Loaded cached CSP+LDA subject-dependent results.")
else:
    csp_sd_results = {}
    for subj in SUBJECTS:
        d = all_data[subj]
        W = fit_csp_ovr(d['X_train'], d['y_train'], n_components=4, n_classes=4)
        feat_train = csp_features(d['X_train'], W)
        feat_test  = csp_features(d['X_test'],  W)

        lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
        lda.fit(feat_train, d['y_train'])
        y_pred = lda.predict(feat_test)

        res = evaluate_csp(d['y_test'], y_pred)
        res['y_pred'] = y_pred.tolist()
        res['y_true'] = d['y_test'].tolist()
        csp_sd_results[subj] = res
        print(f"  {subj}: acc={res['accuracy']:.4f}, κ={res['kappa']:.4f}")

    os.makedirs('metrics', exist_ok=True)
    with open(CSP_SD_PATH, 'wb') as f:
        pickle.dump(csp_sd_results, f)

accs   = [csp_sd_results[s]['accuracy'] for s in SUBJECTS]
kappas = [csp_sd_results[s]['kappa']    for s in SUBJECTS]
print(f"\nCSP+LDA (subject-dep): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")

  A01: acc=0.6437, κ=0.5248
  A02: acc=0.4023, κ=0.2011
  A03: acc=0.7414, κ=0.6550
  A04: acc=0.4425, κ=0.2572
  A05: acc=0.3218, κ=0.0943
  A06: acc=0.4598, κ=0.2798
  A07: acc=0.6379, κ=0.5165
  A08: acc=0.7184, κ=0.6244
  A09: acc=0.6609, κ=0.5477

CSP+LDA (subject-dep): acc=0.559±0.144, κ=0.411±0.192


### 3.2 EEGNet — subject-dependent

In [13]:
# Training hyperparameters (same as Notebook 5)
EEGNET_EPOCHS = 300
EEGNET_LR = 1e-3
BATCH_SIZE = 64

In [14]:
EEGNET_SD_METRICS = 'metrics/capstone_eegnet_subject_dependent.pt'

if os.path.exists(EEGNET_SD_METRICS):
    eegnet_sd_results = torch.load(EEGNET_SD_METRICS, weights_only=False)
    print("Loaded cached EEGNet subject-dependent results.")
    for subj in SUBJECTS:
        r = eegnet_sd_results[subj]
        print(f"  {subj}: acc={r['accuracy']:.4f}, κ={r['kappa']:.4f}")
else:
    eegnet_sd_results = {}
    os.makedirs('metrics', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    for subj in SUBJECTS:
        print(f"\n{'='*50}")
        print(f"EEGNet — Subject {subj}")
        print(f"{'='*50}")

        d = all_data[subj]
        train_loader = DataLoader(EEGDataset(d['X_train'], d['y_train']),
                                  batch_size=BATCH_SIZE, shuffle=True)
        test_loader  = DataLoader(EEGDataset(d['X_test'], d['y_test']),
                                  batch_size=BATCH_SIZE, shuffle=False)

        model = EEGNet().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=EEGNET_LR)

        for epoch in range(1, EEGNET_EPOCHS + 1):
            loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
            if epoch % 100 == 0:
                acc, kappa, _, _ = evaluate_model(model, test_loader, DEVICE)
                print(f"  Epoch {epoch}/{EEGNET_EPOCHS} | loss: {loss:.4f} | acc: {acc:.4f} | κ: {kappa:.4f}")

        acc, kappa, preds, labels = evaluate_model(model, test_loader, DEVICE)
        eegnet_sd_results[subj] = {
            'accuracy': float(acc),
            'kappa': float(kappa),
            'y_pred': preds.tolist(),
            'y_true': labels.tolist()
        }

        torch.save(model.state_dict(), f'models/capstone_eegnet_sd_{subj}.pt')

    torch.save(eegnet_sd_results, EEGNET_SD_METRICS)
    print("\nSaved EEGNet subject-dependent results.")

accs   = [eegnet_sd_results[s]['accuracy'] for s in SUBJECTS]
kappas = [eegnet_sd_results[s]['kappa']    for s in SUBJECTS]
print(f"\nEEGNet (subject-dep): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")


EEGNet — Subject A01
  Epoch 100/300 | loss: 0.6494 | acc: 0.6149 | κ: 0.4869
  Epoch 200/300 | loss: 0.5051 | acc: 0.6552 | κ: 0.5408
  Epoch 300/300 | loss: 0.4504 | acc: 0.6379 | κ: 0.5181

EEGNet — Subject A02
  Epoch 100/300 | loss: 0.9894 | acc: 0.4310 | κ: 0.2412
  Epoch 200/300 | loss: 0.8467 | acc: 0.4080 | κ: 0.2102


KeyboardInterrupt: 

### 3.3 EEG Conformer — subject-dependent

In [ ]:
# Training hyperparameters (same as Notebook 6)
CONFORMER_EPOCHS = 300
CONFORMER_LR = 5e-4
CONFORMER_WD = 0.01

In [ ]:
CONFORMER_SD_METRICS = 'metrics/capstone_conformer_subject_dependent.pt'

if os.path.exists(CONFORMER_SD_METRICS):
    conformer_sd_results = torch.load(CONFORMER_SD_METRICS, weights_only=False)
    print("Loaded cached Conformer subject-dependent results.")
    for subj in SUBJECTS:
        r = conformer_sd_results[subj]
        print(f"  {subj}: acc={r['accuracy']:.4f}, κ={r['kappa']:.4f}")
else:
    conformer_sd_results = {}
    os.makedirs('metrics', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    for subj in SUBJECTS:
        print(f"\n{'='*50}")
        print(f"Conformer — Subject {subj}")
        print(f"{'='*50}")

        d = all_data[subj]
        train_loader = DataLoader(EEGDataset(d['X_train'], d['y_train']),
                                  batch_size=BATCH_SIZE, shuffle=True)
        test_loader  = DataLoader(EEGDataset(d['X_test'], d['y_test']),
                                  batch_size=BATCH_SIZE, shuffle=False)

        model = EEGConformer().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CONFORMER_LR, weight_decay=CONFORMER_WD)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFORMER_EPOCHS)

        for epoch in range(1, CONFORMER_EPOCHS + 1):
            loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
            scheduler.step()
            if epoch % 100 == 0:
                acc, kappa, _, _ = evaluate_model(model, test_loader, DEVICE)
                print(f"  Epoch {epoch}/{CONFORMER_EPOCHS} | loss: {loss:.4f} | acc: {acc:.4f} | κ: {kappa:.4f}")

        acc, kappa, preds, labels = evaluate_model(model, test_loader, DEVICE)
        conformer_sd_results[subj] = {
            'accuracy': float(acc),
            'kappa': float(kappa),
            'y_pred': preds.tolist(),
            'y_true': labels.tolist()
        }

        torch.save(model.state_dict(), f'models/capstone_conformer_sd_{subj}.pt')

    torch.save(conformer_sd_results, CONFORMER_SD_METRICS)
    print("\nSaved Conformer subject-dependent results.")

accs   = [conformer_sd_results[s]['accuracy'] for s in SUBJECTS]
kappas = [conformer_sd_results[s]['kappa']    for s in SUBJECTS]
print(f"\nConformer (subject-dep): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")

### 3.4 Subject-dependent summary

In [ ]:
# Summary table — subject-dependent results
sd_models = {
    'CSP+LDA':       csp_sd_results,
    'EEGNet':        eegnet_sd_results,
    'EEG Conformer': conformer_sd_results,
}

print(f"{'Model':<18} {'Accuracy':>16} {'Kappa':>16}")
print('=' * 52)
for name, res in sd_models.items():
    accs   = [res[s]['accuracy'] for s in SUBJECTS]
    kappas = [res[s]['kappa']    for s in SUBJECTS]
    print(f"{name:<18} {np.mean(accs):.3f} ± {np.std(accs):.3f}   {np.mean(kappas):.3f} ± {np.std(kappas):.3f}")

---

## Part 4 — Subject-independent evaluation (LOSO)

### 4.1 What is leave-one-subject-out (LOSO)?

In subject-dependent evaluation, you train and test on the *same* subject. This simulates a BCI that requires per-user calibration — the user sits through a recording session before the system works.

In **subject-independent** (LOSO) evaluation, you train on data from all subjects *except one*, and test on the held-out subject. You repeat this for all 9 subjects and report the mean. This simulates a **zero-calibration BCI** — a new user puts on the headset and the system works immediately.

Concretely, for fold $k$ (testing on subject $k$):

$$X_{\text{train}} = \bigcup_{j \neq k} X_j^{\text{train}}, \quad X_{\text{test}} = X_k^{\text{test}}$$

where $X_j^{\text{train}}$ is the training split of subject $j$'s data (from `run_pipeline()`), and $X_k^{\text{test}}$ is the test split of the held-out subject.

We use each subject's *training windows* for the pooled training set, and each subject's *test windows* for evaluation. This avoids any data leakage — the held-out subject's test windows were never seen during training.

> **Why is LOSO harder?** EEG varies enormously across individuals — skull thickness, cortical folding, electrode impedances, and cognitive strategy all differ. CSP filters optimized on 8 subjects may not work for the 9th because the spatial pattern of mu-rhythm suppression is anatomically different. Deep models can potentially learn *subject-invariant* features that generalize, but only if the training set is large and diverse enough.

> **Why does this matter for products?** A consumer BCI that requires 20 minutes of calibration per session has terrible UX. The path to a viable product is closing the subject-dependent → subject-independent performance gap. This is an active research problem in BCI.

### Dimension check

With 9 subjects and ~690 training windows each, the pooled training set for one LOSO fold has ~5,520 windows — about 8× more than subject-dependent training. The test set is still ~174 windows (one subject's test split).

For CSP + LDA, the feature extraction is identical — `fit_csp_ovr` on the pooled data, then `csp_features` on the test data. The covariance matrices are larger (more trials), but CSP remains a closed-form eigenvalue problem.

For deep models, the larger training set means the DataLoader produces ~86 batches per epoch (vs ~11), so each epoch is ~8× slower. But 300 epochs should still converge — you may even find the models converge faster because they see more diverse examples per epoch.

### Task 4.1 — CSP + LDA under LOSO

Implement the LOSO loop for CSP + LDA. For each held-out subject:
1. Pool `X_train` from all other subjects into one big array.
2. Pool `y_train` correspondingly.
3. Fit CSP + LDA on the pooled data.
4. Evaluate on the held-out subject's `X_test`, `y_test`.

In [ ]:
CSP_SI_PATH = 'metrics/capstone_csp_lda_subject_independent.pkl'

if os.path.exists(CSP_SI_PATH):
    with open(CSP_SI_PATH, 'rb') as f:
        csp_si_results = pickle.load(f)
    print("Loaded cached CSP+LDA LOSO results.")
else:
    csp_si_results = {}

    for test_subj in SUBJECTS:
        # YOUR CODE HERE
        # 1. Pool X_train and y_train from all subjects EXCEPT test_subj
        # 2. Fit CSP (fit_csp_ovr) on pooled training data
        # 3. Extract features for pooled train and test_subj's X_test
        # 4. Fit LDA on pooled features, predict on test features
        # 5. Store results
        raise NotImplementedError

    os.makedirs('metrics', exist_ok=True)
    with open(CSP_SI_PATH, 'wb') as f:
        pickle.dump(csp_si_results, f)

accs   = [csp_si_results[s]['accuracy'] for s in SUBJECTS]
kappas = [csp_si_results[s]['kappa']    for s in SUBJECTS]
print(f"\nCSP+LDA (LOSO): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")

### Task 4.2 — EEGNet under LOSO

Same structure, but now you need to pool the `EEGDataset` objects from 8 subjects into one DataLoader. Use `torch.utils.data.ConcatDataset` to combine them.

**`ConcatDataset` usage:**
```python
from torch.utils.data import ConcatDataset
pooled = ConcatDataset([ds1, ds2, ds3, ...])
```
This creates a single Dataset that indexes across all constituent datasets. The DataLoader then shuffles across the entire pooled set.

In [ ]:
EEGNET_SI_METRICS = 'metrics/capstone_eegnet_subject_independent.pt'

if os.path.exists(EEGNET_SI_METRICS):
    eegnet_si_results = torch.load(EEGNET_SI_METRICS, weights_only=False)
    print("Loaded cached EEGNet LOSO results.")
    for subj in SUBJECTS:
        r = eegnet_si_results[subj]
        print(f"  {subj}: acc={r['accuracy']:.4f}, κ={r['kappa']:.4f}")
else:
    eegnet_si_results = {}
    os.makedirs('metrics', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    for test_subj in SUBJECTS:
        print(f"\n{'='*50}")
        print(f"EEGNet LOSO — test subject: {test_subj}")
        print(f"{'='*50}")

        # YOUR CODE HERE
        # 1. Create EEGDataset for each training subject (all except test_subj)
        # 2. Combine with ConcatDataset
        # 3. Create train_loader from pooled dataset, test_loader from test_subj
        # 4. Train EEGNet for EEGNET_EPOCHS
        # 5. Evaluate and store results
        raise NotImplementedError

    torch.save(eegnet_si_results, EEGNET_SI_METRICS)
    print("\nSaved EEGNet LOSO results.")

accs   = [eegnet_si_results[s]['accuracy'] for s in SUBJECTS]
kappas = [eegnet_si_results[s]['kappa']    for s in SUBJECTS]
print(f"\nEEGNet (LOSO): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")

### Task 4.3 — EEG Conformer under LOSO

In [ ]:
CONFORMER_SI_METRICS = 'metrics/capstone_conformer_subject_independent.pt'

if os.path.exists(CONFORMER_SI_METRICS):
    conformer_si_results = torch.load(CONFORMER_SI_METRICS, weights_only=False)
    print("Loaded cached Conformer LOSO results.")
    for subj in SUBJECTS:
        r = conformer_si_results[subj]
        print(f"  {subj}: acc={r['accuracy']:.4f}, κ={r['kappa']:.4f}")
else:
    conformer_si_results = {}
    os.makedirs('metrics', exist_ok=True)
    os.makedirs('models', exist_ok=True)

    for test_subj in SUBJECTS:
        print(f"\n{'='*50}")
        print(f"Conformer LOSO — test subject: {test_subj}")
        print(f"{'='*50}")

        # YOUR CODE HERE
        # Same structure as EEGNet LOSO, but use EEGConformer, AdamW, CosineAnnealingLR
        raise NotImplementedError

    torch.save(conformer_si_results, CONFORMER_SI_METRICS)
    print("\nSaved Conformer LOSO results.")

accs   = [conformer_si_results[s]['accuracy'] for s in SUBJECTS]
kappas = [conformer_si_results[s]['kappa']    for s in SUBJECTS]
print(f"\nConformer (LOSO): acc={np.mean(accs):.3f}±{np.std(accs):.3f}, κ={np.mean(kappas):.3f}±{np.std(kappas):.3f}")

### Decision — Subject-dependent vs. subject-independent

1. Which model suffered the largest drop in kappa going from subject-dependent to LOSO? Which suffered the least? Does this match your expectations about which models should transfer better?

2. CSP fits spatial filters from the data covariance. In LOSO, the pooled covariance averages over 8 subjects with different head anatomies. Why might this *hurt* CSP more than it hurts EEGNet?

3. The pooled training set has ~5,500 windows (8× more than subject-dependent). Did the deep models benefit from this larger dataset? If not, what does that tell you about the nature of cross-subject variability in EEG — is it primarily a *data quantity* problem or a *domain shift* problem?

*Write your answers here.*

---

## Part 5 — Confusion matrix analysis

### 5.1 Why confusion matrices matter for BCI

Accuracy and kappa are scalar summaries — they don't tell you *which* classes are confused. For a BCI product, this matters: if left hand and right hand are confused with each other but never with feet, you might design a 2-class left/right BCI and ignore the other classes. Or you might add a secondary confirmation step for ambiguous predictions.

**Neurophysiological prediction:** Left hand and right hand motor imagery produce *contralateral* mu-rhythm suppression — left hand suppresses the right sensorimotor cortex (around C4) and vice versa. These are mirror-image patterns on the same frequency band, making them the hardest pair to separate. Feet imagery produces suppression at the vertex (Cz), which is spatially distinct. Tongue imagery is the least studied and may produce the weakest ERD signal.

### Task 5.1 — Plot confusion matrices

Create a 3×2 grid of confusion matrices: 3 models × 2 paradigms. Aggregate predictions across all 9 subjects (concatenate all `y_true` and `y_pred`).

In [ ]:
# YOUR CODE HERE
# 1. For each model × paradigm, aggregate y_true and y_pred across all 9 subjects
# 2. Compute confusion_matrix(y_true_all, y_pred_all, normalize='true')
#    normalize='true' gives per-class recall rates (each row sums to 1)
# 3. Plot a 3×2 grid using ConfusionMatrixDisplay
#    Rows: CSP+LDA, EEGNet, Conformer
#    Columns: Subject-dependent, Subject-independent (LOSO)
#    Use display_labels=CLASS_NAMES

raise NotImplementedError

### Observation — Confusion patterns

> Across all models, the most-confused class pair is ___ and ___. The neurophysiological explanation is ___.
>
> The class with the highest recall across all models is ___. This makes sense because ___.
>
> Going from subject-dependent to LOSO, the class that suffers the largest recall drop is ___. This suggests that ___.
>
> If I were designing a 2-class BCI for maximum accuracy, I would use classes ___ and ___ because ___.

*Write your answers here.*

---

## Part 6 — Per-subject performance analysis

### 6.1 Which subjects are easy, which are hard?

BCI performance varies dramatically across subjects — this is called **BCI illiteracy** (or more recently, **BCI inefficiency**). Some people produce strong, consistent ERD patterns; others produce weak or variable signals. A production BCI system needs to handle both.

### Task 6.1 — Per-subject comparison plot

Create a grouped bar chart showing kappa for each subject × model × paradigm.

In [ ]:
# YOUR CODE HERE
# Create a grouped bar chart:
# X-axis: 9 subjects
# Groups of 6 bars per subject: CSP-SD, EEGNet-SD, Conformer-SD, CSP-LOSO, EEGNet-LOSO, Conformer-LOSO
# Y-axis: Cohen's kappa
# Add a horizontal line at kappa=0 (chance) and kappa=0.4 ("fair" agreement)
# Use distinct colors for each model, solid vs hatched for SD vs LOSO

raise NotImplementedError

### Observation — Per-subject analysis

> The "easiest" subject is ___ (highest mean kappa across models). The "hardest" is ___.
>
> The performance gap between the best and worst subject is ___ kappa points. This is ___ (larger/smaller) than the gap between models on the same subject.
>
> For the hardest subject, the best model achieves κ = ___. This is ___ (above/below) the "fair agreement" threshold of 0.4. In a deployed system, this means ___.
>
> The model that is most *consistent* across subjects (lowest std) is ___. This matters for a product because ___.

*Write your answers here.*

---

## Part 7 — Model explainability via saliency maps

### 7.1 What are saliency maps?

A **saliency map** shows which parts of the input are most important for the model's prediction. For EEG, this means: which channels and which timepoints does the model pay attention to?

The simplest saliency method is **input gradients**: compute the gradient of the predicted class score with respect to the input, then take the absolute value. If $f_c(x)$ is the logit for class $c$ and $x$ is the input:

$$S(x) = \left| \frac{\partial f_c(x)}{\partial x} \right|$$

- $f_c(x)$: the raw logit (before softmax) for the predicted class $c$
- $x$: the input tensor, shape `(1, 1, C, T)` — one EEG window
- $S(x)$: the saliency map, same shape as $x$

High values in $S(x)$ mean that small changes to that input element strongly affect the prediction. For EEG, we expect high saliency:
- Over **sensorimotor channels** (C3, C4, Cz) — where motor imagery ERD occurs
- During the **active imagery period** — the time window when the subject is imagining movement

If the saliency map instead highlights edge channels (like POz or Fz) or is uniformly spread, the model may be learning noise or artifacts rather than neurophysiological signal.

### Implementation

**Step 1 — Compute saliency for one trial:**
```
1. Set model to eval mode
2. Create input tensor x with requires_grad=True
3. Forward pass: logits = model(x)
4. Get predicted class: c = argmax(logits)
5. Backward pass: logits[0, c].backward()
6. Saliency = |x.grad|, squeeze to (C, T)
```

**Step 2 — Average over many trials per class:**
Compute saliency for multiple correctly-classified test trials of each class, then average. This reduces noise and reveals the consistent pattern.

**Step 3 — Visualize as a heatmap:**
Plot a (C, T) heatmap with channel names on the y-axis and time on the x-axis. Highlight channels C3, C4, Cz.

> **Why saliency and not attention weights?** The Conformer has attention weights you could visualize, but those only show temporal attention within the Transformer — they don't tell you about channel importance (that's in the CNN front-end). Saliency maps work for *any* differentiable model and capture the full computation, CNN + Transformer combined.

### Task 7.1 — Implement saliency computation

Write a function that computes the saliency map for a single trial.

In [ ]:
def compute_saliency(model, x, device):
    """
    Compute input gradient saliency for one trial.

    Args:
        model: nn.Module (in eval mode)
        x: np.ndarray, shape (C, T) — one raw EEG window
        device: torch.device

    Returns:
        saliency: np.ndarray, shape (C, T) — |d logit_c / d x|
        pred_class: int — predicted class index

    Steps:
        1. Convert x to tensor (1, 1, C, T) with requires_grad=True
        2. Forward pass
        3. Backward on the predicted class logit
        4. Return |x.grad| squeezed to (C, T)
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Sanity check — saliency shape and non-zero gradients
# Load a trained EEGNet model for A01 (subject-dependent)
_model = EEGNet().to(DEVICE)
_model.load_state_dict(torch.load('models/capstone_eegnet_sd_A01.pt', map_location=DEVICE))
_model.eval()

_x_sample = all_data['A01']['X_test'][0]  # shape (C, T)
_sal, _pred = compute_saliency(_model, _x_sample, DEVICE)

assert _sal.shape == (22, 250), f"Expected (22, 250), got {_sal.shape}"
assert _sal.sum() > 0, "Saliency is all zeros — backward() may not have computed gradients"
print(f"✓ Saliency shape: {_sal.shape}, predicted class: {_pred}")
print(f"  Saliency range: [{_sal.min():.6f}, {_sal.max():.6f}]")
del _model

### Task 7.2 — Class-averaged saliency maps

For each class, compute saliency on up to 30 correctly-classified test trials, then average. Do this for both EEGNet and the Conformer (subject-dependent, A01).

Plot a 2×4 grid of heatmaps: 2 models × 4 classes.

In [ ]:
def compute_class_averaged_saliency(model, X_test, y_test, y_pred, device, max_trials=30):
    """
    Compute class-averaged saliency maps.

    Args:
        model: nn.Module (trained)
        X_test: np.ndarray, shape (N, C, T)
        y_test: np.ndarray, shape (N,) — true labels
        y_pred: np.ndarray, shape (N,) — predicted labels
        device: torch.device
        max_trials: int — max trials per class to average over

    Returns:
        saliency_maps: dict mapping class_idx → np.ndarray of shape (C, T)
    """
    # YOUR CODE HERE
    # For each class c in range(4):
    #   1. Find indices where y_test == c AND y_pred == c (correctly classified)
    #   2. Take up to max_trials of them
    #   3. Compute saliency for each, stack, average
    #   4. Store in saliency_maps[c]
    raise NotImplementedError

In [ ]:
# Compute saliency maps for EEGNet and Conformer (subject-dependent, A01)
# Load trained models
eegnet_a01 = EEGNet().to(DEVICE)
eegnet_a01.load_state_dict(torch.load('models/capstone_eegnet_sd_A01.pt', map_location=DEVICE))
eegnet_a01.eval()

conformer_a01 = EEGConformer().to(DEVICE)
conformer_a01.load_state_dict(torch.load('models/capstone_conformer_sd_A01.pt', map_location=DEVICE))
conformer_a01.eval()

X_test_a01 = all_data['A01']['X_test']
y_test_a01 = all_data['A01']['y_test']

# Get predictions
eegnet_preds = np.array(eegnet_sd_results['A01']['y_pred'])
conformer_preds = np.array(conformer_sd_results['A01']['y_pred'])

print("Computing EEGNet saliency maps...")
eegnet_saliency = compute_class_averaged_saliency(
    eegnet_a01, X_test_a01, y_test_a01, eegnet_preds, DEVICE)

print("Computing Conformer saliency maps...")
conformer_saliency = compute_class_averaged_saliency(
    conformer_a01, X_test_a01, y_test_a01, conformer_preds, DEVICE)

print("Done.")

In [ ]:
# Plot saliency heatmaps — 2 models × 4 classes
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

# Channel indices for sensorimotor channels (highlight these)
# C3=index 7, C4=index 11, Cz=index 9
sensorimotor_idx = [7, 9, 11]

time_axis = np.arange(N_TIMEPOINTS) / FS  # seconds

for col, class_idx in enumerate(range(4)):
    for row, (name, sal_dict) in enumerate([('EEGNet', eegnet_saliency),
                                             ('Conformer', conformer_saliency)]):
        ax = axes[row, col]
        sal = sal_dict[class_idx]

        im = ax.imshow(sal, aspect='auto', cmap='hot',
                       extent=[0, 1, N_CHANNELS - 0.5, -0.5])
        ax.set_yticks(range(N_CHANNELS))
        ax.set_yticklabels(CH_NAMES, fontsize=6)

        # Highlight sensorimotor channels
        for idx in sensorimotor_idx:
            ax.axhline(y=idx, color='cyan', linewidth=0.5, alpha=0.5)

        ax.set_title(f"{name} — {CLASS_NAMES[class_idx]}", fontsize=10)
        if row == 1:
            ax.set_xlabel('Time (s)')
        if col == 0:
            ax.set_ylabel('Channel')

plt.suptitle('Saliency Maps (Subject A01, Subject-Dependent)', fontsize=14)
plt.tight_layout()
plt.show()

### Decision — Saliency interpretation

1. For **Left Hand** imagery, which channels have the highest saliency in EEGNet? Do they correspond to the contralateral hemisphere (right side — around C4)? What about Right Hand imagery — do you see the mirror pattern?

2. For **Feet** imagery, saliency should concentrate around the vertex (Cz). Does it? This is because foot motor cortex is located medially, at the top of the brain.

3. Compare the saliency patterns between EEGNet and the Conformer. Are they looking at the same channels? If one model has more diffuse saliency (spread over many channels), what might that tell you about how it's making decisions?

4. If saliency is concentrated at the *edges* of the time window (near t=0 or t=1s), this might indicate the model is learning onset/offset transients rather than sustained ERD. Do you see this? What are the implications for a real-time BCI?

*Write your answers here.*

---

## Part 8 — Performance drop analysis

### 8.1 Quantifying the calibration gap

The **calibration gap** is the performance difference between subject-dependent and subject-independent evaluation. This single number captures how much a BCI system loses when deployed without per-user calibration.

$$\Delta\kappa = \kappa_{\text{subject-dep}} - \kappa_{\text{LOSO}}$$

A model that minimizes $\Delta\kappa$ is more deployable. A model with low $\kappa_{\text{LOSO}}$ but also low $\Delta\kappa$ might simply be bad at everything — what we want is high $\kappa_{\text{LOSO}}$ *and* low $\Delta\kappa$.

### Task 8.1 — Compute and visualize the calibration gap

In [ ]:
# YOUR CODE HERE
# For each model:
# 1. Compute per-subject Δκ = κ_SD - κ_LOSO
# 2. Report mean ± std of the gap
# 3. Create a bar chart showing Δκ per subject × model

models_sd = {'CSP+LDA': csp_sd_results, 'EEGNet': eegnet_sd_results, 'Conformer': conformer_sd_results}
models_si = {'CSP+LDA': csp_si_results, 'EEGNet': eegnet_si_results, 'Conformer': conformer_si_results}

print(f"{'Model':<18} {'κ (SD)':>10} {'κ (LOSO)':>10} {'Δκ':>10}")
print('=' * 50)

raise NotImplementedError

### Observation — Calibration gap

> The model with the smallest calibration gap (Δκ) is ___, with Δκ = ___ ± ___.
>
> The model with the largest gap is ___. This is because ___.
>
> For subjects A01 and A03 (typically the "easiest" subjects), the gap is ___. For the hardest subjects (A04, A05), the gap is ___. This tells us that ___.
>
> If I were building a consumer BCI product, I would choose ___ as the base model because ___.

*Write your answers here.*

---

## Part 9 — Final results table

This is the deliverable — the table that goes in your project report.

### Task 9.1 — Comprehensive results table

In [ ]:
# ============================================================
# FINAL RESULTS TABLE
# ============================================================

print("="*80)
print("BCI Competition IV Dataset 2a — Motor Imagery Classification Results")
print("="*80)

# Table 1: Mean ± std across 9 subjects
print("\nTable 1: Mean performance across 9 subjects")
print(f"{'Model':<18} {'Paradigm':<16} {'Accuracy':>16} {'Kappa':>16}")
print('-' * 68)

for name in ['CSP+LDA', 'EEGNet', 'EEG Conformer']:
    sd_res = models_sd[name] if name != 'EEG Conformer' else models_sd['Conformer']
    si_res = models_si[name] if name != 'EEG Conformer' else models_si['Conformer']

    for paradigm, res in [('Subject-dep', sd_res), ('LOSO', si_res)]:
        accs   = [res[s]['accuracy'] for s in SUBJECTS]
        kappas = [res[s]['kappa']    for s in SUBJECTS]
        print(f"{name:<18} {paradigm:<16} {np.mean(accs):.3f} ± {np.std(accs):.3f}   {np.mean(kappas):.3f} ± {np.std(kappas):.3f}")
    print()

# Table 2: Per-subject kappa (subject-dependent)
print("\nTable 2: Per-subject kappa (subject-dependent)")
print(f"{'Subject':<10}", end='')
for name in ['CSP+LDA', 'EEGNet', 'Conformer']:
    print(f"{name:>12}", end='')
print()
print('-' * 46)
for subj in SUBJECTS:
    print(f"{subj:<10}", end='')
    for name in ['CSP+LDA', 'EEGNet', 'Conformer']:
        res = models_sd[name]
        print(f"{res[subj]['kappa']:>12.3f}", end='')
    print()

# Table 3: Per-subject kappa (LOSO)
print("\nTable 3: Per-subject kappa (LOSO)")
print(f"{'Subject':<10}", end='')
for name in ['CSP+LDA', 'EEGNet', 'Conformer']:
    print(f"{name:>12}", end='')
print()
print('-' * 46)
for subj in SUBJECTS:
    print(f"{subj:<10}", end='')
    for name in ['CSP+LDA', 'EEGNet', 'Conformer']:
        res = models_si[name]
        print(f"{res[subj]['kappa']:>12.3f}", end='')
    print()

### Task 9.2 — Publication-quality results figure

Create a single figure with two subplots side by side:
- Left: subject-dependent kappas (3 models, 9 subjects as grouped bars)
- Right: LOSO kappas (same layout)

Include error bars (mean ± std as horizontal dashed lines), chance level, and a "fair" agreement line at κ = 0.4.

In [ ]:
# YOUR CODE HERE
# Create a publication-quality comparison figure
# Two subplots: subject-dependent (left) vs LOSO (right)
# Grouped bars: 3 models per subject
# Horizontal reference lines at κ=0 (chance) and κ=0.4 (fair)

raise NotImplementedError

---

## Part 10 — Save all results for reporting

In [ ]:
# Save comprehensive results dict
capstone_results = {
    'subject_dependent': {
        'CSP+LDA':       csp_sd_results,
        'EEGNet':        eegnet_sd_results,
        'EEG Conformer': conformer_sd_results,
    },
    'subject_independent': {
        'CSP+LDA':       csp_si_results,
        'EEGNet':        eegnet_si_results,
        'EEG Conformer': conformer_si_results,
    },
    'dataset': 'BCI Competition IV Dataset 2a',
    'n_subjects': 9,
    'n_classes': 4,
    'window_size_s': 1.0,
    'sampling_rate': 250,
}

with open('metrics/capstone_all_results.pkl', 'wb') as f:
    pickle.dump(capstone_results, f)

print("All capstone results saved to metrics/capstone_all_results.pkl")

---

## Part 11 — Final reflections

### What you should be able to do now

1. Design and execute a **subject-dependent vs. subject-independent** evaluation — the standard comparison framework in BCI research.
2. Implement **LOSO cross-validation** by pooling multi-subject EEG data into a single training set while keeping the held-out subject's data separate.
3. Compute and interpret **saliency maps** (input gradients) to verify that a neural network is learning neurophysiologically meaningful features from EEG.
4. Analyze **confusion matrices** across motor imagery classes and connect confusion patterns to the underlying neuroanatomy.
5. Quantify the **calibration gap** — the performance cost of removing per-user calibration — and reason about which model architectures minimize it.

### Decision — Final project reflection

1. Across the entire project, which result surprised you the most? Why?

2. If you had one more week, what single experiment would you run to most improve the LOSO results? (Think about: data augmentation, domain adaptation, fine-tuning, architecture changes, or longer recording windows.)

3. You're pitching a motor imagery BCI startup. A VC asks: "Why can't you just use CSP?" Based on your results, what's your honest answer?

4. The BCI Competition IV Dataset 2a has 288 trials per subject. Modern consumer EEG headsets can record thousands of trials per session. At what training set size do you expect deep models to clearly outperform CSP, and why?

5. How would you adapt this pipeline for a **real-time** BCI that classifies motor imagery continuously (every 250ms, with overlapping windows)? What changes are needed in preprocessing, model design, and evaluation?

*Write your answers here.*